<h2 style="color: #ff00aa; font-size: 26px; text-align: center; margin-top: 20px; margin-bottom: 5px; font-family: 'Segoe UI', sans-serif; font-weight: 600;">2D Elemental Mapping Visualization</h2>
<p style="color: #ff00aa; text-align: center; margin-top: 0; margin-bottom: 25px; font-size: 14px; font-weight: bold; text-transform: uppercase; letter-spacing: 1px;"> José Sousa-Brito</p>

<h2 style="color: #ff00aa; font-size: 20px; font-family: 'Segoe UI', sans-serif; font-weight: 600; border-bottom: 2px solid #ff00aa; padding-bottom: 5px;">Pipeline Core Architecture</h2>

This pipeline automates the batch visualization of 2D elemental distribution maps by converting raw intensity matrices exported from the analytical equipment into vector graphics:

| Functional Stage | Operational Protocol |
| :--- | :--- |
| **Interactive Folder Selection** | Launches a native OS file dialog (`tkinter`) allowing the user to browse and select the target directory containing the raw `.csv` matrices. |
| **Scan Area Input** | Prompts the user for the physical dimensions of the scanned region (in µm) via a dialog box, pre-filled with a default value for convenience. |
| **Batch Matrix Loading** | Iterates through every `.csv` file found in the selected folder, loading each one as a raw intensity matrix with `pandas`. |
| **Spatial Rendering** | Plots each matrix with `imshow`, preserving exact pixel spatial resolution with no artificial smoothing or interpolation, scaled to the real-world scan area. |

---

<h2 style="color: #ff00aa; font-size: 20px; font-family: 'Segoe UI', sans-serif; font-weight: 600; border-bottom: 2px solid #ff00aa; padding-bottom: 5px;">How to Use the Code</h2>

When executing the script, an automated native file dialog will prompt you to select the target directory containing your raw data matrices.

Immediately after, an input dialog window will ask you to provide the physical dimensions of the scanned region of interest (in micrometers), with a default value pre-filled for convenience.

The pipeline dynamically scans the selected folder, extracts all standard `.csv` files exported from the analytical equipment, and processes them sequentially using the custom area size.

The code automatically maps the intensity counts using the exact pixel spatial resolution without any artificial smoothing, saving the PDFs directly into the source data directory.

### Output

For every input file, the script exports a vector-based, transparent PDF map directly into the source directory, appending a dedicated suffix to preserve the raw data integrity:

```python
output_filename = f"{file_path.stem}_peaks.pdf"

In [ ]:
import os
import glob
from pathlib import Path
import tkinter as tk
from tkinter import filedialog, simpledialog
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def select_folder_and_run_pipeline():
    root = tk.Tk()
    root.withdraw() 
    root.attributes('-topmost', True) 
    root.update()
    
    print("Please select the folder containing your raw CSV matrices...")
    selected_dir = filedialog.askdirectory(title="Select Folder with CSV Maps")
    
    if not selected_dir:
        print("No folder selected. Exiting pipeline.")
        return
        
    folder_path = Path(selected_dir)
    csv_files = list(folder_path.glob("*.csv"))
    
    if not csv_files:
        print(f"No .csv files found in: {folder_path}")
        return
        
    area_microns = simpledialog.askfloat("Input Parameter", 
                                         "Enter scan area size (in µm):", 
                                         initialvalue=2640)
    
    if area_microns is None:
        print("No scan area provided. Exiting pipeline.")
        return
        
    print(f"Target scan area set to: {area_microns} µm")
    print(f"Found {len(csv_files)} CSV file(s). Starting batch visualization...\n")
    
    for file_path in csv_files:
        print(f"Processing: {file_path.name}")
        
        df = pd.read_csv(file_path, header=None)
        matrix = df.values
        
        fig = plt.figure(figsize=(15, 12), dpi=300, facecolor='none')
        ax = fig.add_subplot(111, facecolor='none')
        
        im = ax.imshow(matrix,   
                       cmap='gist_ncar',   
                       origin='lower',   
                       interpolation='nearest',   
                       extent=[0, area_microns, 0, area_microns])
        
        ax.set_xlabel(r'Distance ($\mu$m)', fontsize=35, fontweight='bold')
        ax.set_ylabel(r'Distance ($\mu$m)', fontsize=35, fontweight='bold')
        
        for spine in ax.spines.values():
            spine.set_linewidth(2)
            
        cbar = fig.colorbar(im, pad=0.02, aspect=30)
        cbar.set_label('Intensity (Counts)', fontsize=35, labelpad=15, fontweight='bold')
        
        ax.tick_params(axis='both', labelsize=28, width=2)   
        cbar.ax.tick_params(labelsize=28)                  
        
        output_filename = f"{file_path.stem}_ORIGINAL.pdf"
        output_path = folder_path / output_filename
        
        print(f"Saving vector map to: {output_path}")
        
        plt.savefig(output_path,   
                    bbox_inches='tight',   
                    transparent=True,   
                    dpi=300)
        
        plt.show()

        plt.close(fig)
        print(f"Successfully generated {output_filename}\n")
        
    print("Batch processing complete! All maps have been saved to the target directory.")

if __name__ == "__main__":
    select_folder_and_run_pipeline()